# Entity Comparison API — manual test

Posts location master data + an entity payload, then inspects what the dynamic rule
engine wrote to `validation_results`. Validation now runs against the **join** of
`entities` and `entities_location` on `(systemCode, businessEntityCode)`.

**Before running:**
1. `python seed_validation_rules.py`  (one-time, loads the rule set)
2. `python run.py`  (starts the API on http://localhost:8000)
3. Run `reset_db.ipynb` for a clean slate (every POST stores new rows, so re-runs pile
   up duplicates, and `entities_location` has no de-dup).

**Order matters:** locations must be POSTed **before** the entities, because the
join happens synchronously while each entity is inserted. Posting entities first
means the rule engine sees no `Country` and every location-based rule passes by
default.

Test data files: `sample_locations.json`, `sample_payload.json` — edit those to
change the inputs.

In [41]:
# pip install requests pandas  # uncomment if needed
import os
import json
import requests
import pandas as pd

BASE_URL = "http://localhost:8000"

# resolve the project root (folder containing run.py) so relative paths work
# no matter where the kernel was started
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here

LOCATIONS_FILE = os.path.join(PROJECT_ROOT, "sample_locations.json")
PAYLOAD_FILE = os.path.join(PROJECT_ROOT, "sample_payload copy 2.json")

print("project root:", PROJECT_ROOT)
print("liveness    :", requests.get(f"{BASE_URL}/docs").status_code)

project root: d:\Python\Event Hub
liveness    : 200


## 1. POST location master data  (run this first)

In [35]:
def post_locations(locations, base_url=BASE_URL):
    """POST a list of location master-data records to /entities/inputlocation.

    Each item must carry `systemCode`, `Code` (-> businessEntityCode) and
    `Country`; `Name`, `Type`, `Alternate_code`, `City`, `Zip` are optional.
    Returns the endpoint's per-row summary list.
    """
    if isinstance(locations, dict):
        locations = [locations]
    resp = requests.post(f"{base_url}/entities/inputlocation", json=locations)
    resp.raise_for_status()
    return resp.json()


with open(LOCATIONS_FILE, "r", encoding="utf-8") as f:
    locations = json.load(f)

print(f"loaded {len(locations)} location(s) from {LOCATIONS_FILE}")
location_result = post_locations(locations)
print(json.dumps(location_result, indent=2))

loaded 5 location(s) from d:\Python\Event Hub\sample_locations.json
[
  {
    "code": "SGLN_mapping",
    "country": "SG"
  },
  {
    "code": "DOCS99",
    "country": "MY"
  },
  {
    "code": "BG01",
    "country": "BG"
  },
  {
    "code": "BG02",
    "country": "DE"
  },
  {
    "code": "BGN1",
    "country": "JP"
  }
]


In [36]:
# confirm what landed in entities_location
locs = requests.get(f"{BASE_URL}/entities/locations").json()
pd.DataFrame(locs)

,id,systemCode,businessEntityCode,name,type,alternate_code,country,city,zip
0,1,ACasdasDC,SGLN_mapping,Ang Mo Kio Distribution Centre,Warehouse,SG-AMK-01,SG,Singapore,569880
1,2,ACDC,DOCS99,Shah Alam Plant,Manufacturing,MY-SA-99,MY,Shah Alam,40150
2,3,ACDC,BG01,Sofia Logistics Hub,Warehouse,BG-SOF-01,BG,Sofia,1000
3,4,ACDC,BG02,Hamburg Cross-Dock,Warehouse,DE-HAM-02,DE,Hamburg,20095
4,5,ACDC,BGN1,Tokyo Bay Depot,Distribution,JP-TYO-1,JP,Tokyo,135-0064
5,6,ACasdasDC,SGLN_mapping,Ang Mo Kio Distribution Centre,Warehouse,SG-AMK-01,SG,Singapore,569880
6,7,ACDC,DOCS99,Shah Alam Plant,Manufacturing,MY-SA-99,MY,Shah Alam,40150
7,8,ACDC,BG01,Sofia Logistics Hub,Warehouse,BG-SOF-01,BG,Sofia,1000
8,9,ACDC,BG02,Hamburg Cross-Dock,Warehouse,DE-HAM-02,DE,Hamburg,20095
9,10,ACDC,BGN1,Tokyo Bay Depot,Distribution,JP-TYO-1,JP,Tokyo,135-0064


## 2. POST the entity payload

In [42]:
with open(PAYLOAD_FILE, "r", encoding="utf-8") as f:
    payload = json.load(f)

mappings = payload.get("mappingsInformation", [payload])
print(f"loaded {len(mappings)} mapping(s) from {PAYLOAD_FILE}")
pd.DataFrame(mappings)

loaded 5 mapping(s) from d:\Python\Event Hub\sample_payload copy 2.json


,systemCode,businessEntityCode,EOID,FID,UKEOID,UKFID,SGLN
0,ACasdqwas,SGLN_mapqweqwping,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0
1,ACDasC,DOCS99,QCLUXXe00000023,QCLUXXf00000023000018,,0ims9SUpP0txasdavB2,urn:epc:id:sgln:71asdjpaksjdp73736.79084.0
2,ACasaDC,BG01,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94asldka;lsdkKFfK9,urn:epc:id:sgln:1233367.11754.0
3,ACasaDC,BG02,LEBGR1e003Tasdp4N3J,,2745,8GrI42lasdooka;sldHnslSc05,urn:epc:id:sgln:1233367.10054.0
4,ACD123C,BGN1,LEBGR1e003Tp4N3J,LEBGR1f0asd0asldasldk5OF84yaBGN1,4068,,urn:epc:id:sgln:1233367.11164.0


In [43]:
resp = requests.post(f"{BASE_URL}/entities/process", json=payload)
print(resp.status_code)
process_result = resp.json()
print(json.dumps(process_result, indent=2))

200
{
  "total_entities": 5,
  "inserted": 5,
  "new_entities": 0,
  "updated_entities": 5,
  "details": [
    {
      "entity_hash": "ef04da778a75f17ea5b6b690c698d7974edae1ce7f8f702c6f037048b13244fb",
      "status": "inserted",
      "id": 41,
      "new_entity": false,
      "businessEntityCode": "SGLN_mapqweqwping"
    },
    {
      "entity_hash": "3f94a784fb62851b506b0c14588449b073a4e8604c0edfe1d21a08a1b7e84d3f",
      "status": "inserted",
      "id": 42,
      "new_entity": false,
      "businessEntityCode": "DOCS99"
    },
    {
      "entity_hash": "19c728348db858d867dd646a0b770086d6995375528adfe8567ee6d8d378aa48",
      "status": "inserted",
      "id": 43,
      "new_entity": false,
      "businessEntityCode": "BG01"
    },
    {
      "entity_hash": "bafd246a473e8a912a681d44341c53fa4e7cdfa234e892190303569d8c74027f",
      "status": "inserted",
      "id": 44,
      "new_entity": false,
      "businessEntityCode": "BG02"
    },
    {
      "entity_hash": "a3cc2721bd8fa7cdf2

## 3. GET /entities  (what got stored)

In [ ]:
entities = requests.get(f"{BASE_URL}/entities").json()
pd.DataFrame(entities)

,id,entity_hash,systemCode,businessEntityCode,EOID,FID,UKEOID,UKFID,SGLN,created_at
0,1,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,ACasdasDC,SGLN_mapping,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-22 03:37:22
1,2,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,ACDC,DOCS99,QCLUXXe00000023,QCLUXXf00000023000018,2429,0ims9SUpP0txasdavB2,urn:epc:id:sgln:7173736.79084.0,2026-09-22 03:37:22
2,3,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,ACDC,BG01,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94KFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-22 03:37:22
3,4,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,ACDC,BG02,LEBGR1e003Tasdp4N3J,LEBGR1f005OF84ya,2745,8GrI42lHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-22 03:37:22
4,5,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,ACDC,BGN1,LEBGR1e003Tp4N3J,LEBGR1f0asd05OF84yaBGN1,4068,Vw2QuxVutMftJ1V,urn:epc:id:sgln:1233367.11164.0,2026-09-22 03:37:22
5,26,ef04da778a75f17ea5b6b690c698d7974edae1ce7f8f70...,ACasdqwas,SGLN_mapqweqwping,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-22 03:41:05
6,27,3f94a784fb62851b506b0c14588449b073a4e8604c0edf...,ACDasC,DOCS99,QCLUXXe00000023,QCLUXXf00000023000018,,0ims9SUpP0txasdavB2,urn:epc:id:sgln:71asdjpaksjdp73736.79084.0,2026-09-22 03:41:05
7,28,19c728348db858d867dd646a0b770086d6995375528adf...,ACasaDC,BG01,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94asldka;lsdkKFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-22 03:41:05
8,29,bafd246a473e8a912a681d44341c53fa4e7cdfa234e892...,ACasaDC,BG02,LEBGR1e003Tasdp4N3J,,2745,8GrI42lasdooka;sldHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-22 03:41:05
9,30,a3cc2721bd8fa7cdf22294cc45b2a42de4533bbaebb9c2...,ACD123C,BGN1,LEBGR1e003Tp4N3J,LEBGR1f0asd0asldasldk5OF84yaBGN1,4068,,urn:epc:id:sgln:1233367.11164.0,2026-09-22 03:41:05


: 

In [40]:
entities = requests.get(f"{BASE_URL}/entities/historical").json()
pd.DataFrame(entities)

,id,entity_hash,systemCode,businessEntityCode,EOID,FID,UKEOID,UKFID,SGLN,created_at
0,1,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,ACasdasDC,SGLN_mapping,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-21 09:51:25
1,2,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,ACDC,DOCS99,QCLUXXe00000023,QCLUXXf00000023000018,2429,0ims9SUpP0txasdavB2,urn:epc:id:sgln:7173736.79084.0,2026-09-21 09:51:25
2,3,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,ACDC,BG01,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94KFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-21 09:51:25
3,4,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,ACDC,BG02,LEBGR1e003Tasdp4N3J,LEBGR1f005OF84ya,2745,8GrI42lHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-21 09:51:25
4,5,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,ACDC,BGN1,LEBGR1e003Tp4N3J,LEBGR1f0asd05OF84yaBGN1,4068,Vw2QuxVutMftJ1V,urn:epc:id:sgln:1233367.11164.0,2026-09-21 09:51:25
5,6,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,ACasdasDC,SGLN_mapping,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-21 09:51:44
6,7,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,ACDC,DOCS99,QCLUXXe00000023,QCLUXXf00000023000018,2429,0ims9SUpP0txasdavB2,urn:epc:id:sgln:7173736.79084.0,2026-09-21 09:51:44
7,8,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,ACDC,BG01,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94KFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-21 09:51:44
8,9,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,ACDC,BG02,LEBGR1e003Tasdp4N3J,LEBGR1f005OF84ya,2745,8GrI42lHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-21 09:51:44
9,10,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,ACDC,BGN1,LEBGR1e003Tp4N3J,LEBGR1f0asd05OF84yaBGN1,4068,Vw2QuxVutMftJ1V,urn:epc:id:sgln:1233367.11164.0,2026-09-21 09:51:44


## 4. GET /entities/validation-results  (entity ⋈ location)

In [10]:
vr = requests.get(f"{BASE_URL}/entities/validation-results").json()
vr_df = pd.DataFrame(vr)
vr_df

,entity_id,entity_hash,businessEntityCode,rule_id,passed,severity,status,description,created_at
0,1,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,SGLN_mapping,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:25
1,2,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,DOCS99,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:25
2,3,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,BG01,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:25
3,4,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,BG02,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:25
4,5,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,BGN1,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:25
5,6,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,SGLN_mapping,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:44
6,7,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,DOCS99,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:44
7,8,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,BG01,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:44
8,9,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,BG02,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:44
9,10,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,BGN1,API-ROMANIZE-NAME,1,Warning,None,None,2026-09-21 09:51:44


In [ ]:
# pass/fail matrix: one row per entity, one column per rule
if not vr_df.empty:
    matrix = vr_df.pivot_table(
        index="businessEntityCode",
        columns="rule_id",
        values="passed",
        aggfunc="last",
    )
    display(matrix.replace({1: "PASS", 0: "FAIL", True: "PASS", False: "FAIL"}))

    print("\nFailures only:")
    display(vr_df[vr_df["passed"].isin([0, False])][
        ["businessEntityCode", "rule_id", "severity", "status", "description"]
    ])

## What to expect

**The join is always applied:** every entity is validated against
`entity_json` **merged with** its `entities_location` row on
`(systemCode, businessEntityCode)`. `get /entities/locations` (cell 1) shows what
was joined in; a missing location row just leaves `Country` / `City` / … blank.

### With the shipped files + the current `seed_validation_rules.py`

Active rules: `VAL-UK-Field`, `FIELD-EXIST`, `FIELD-EOID-Unique`. All three ultimately
test `UKEOID` / `UKFID` / `EOID`, which every row in `sample_payload.json` already
has — so the only failure is the duplicate `EOID`:

| Entity | Joined Country | VAL-UK-Field | FIELD-EXIST | FIELD-EOID-Unique |
|---|---|---|---|---|
| `SGLN_mapping` | `SG` | PASS | PASS | PASS |
| `DOCS99` | `MY` | PASS | PASS | PASS |
| `BG01` | `BG` | PASS | PASS | PASS |
| `BG02` | `DE` | PASS | PASS | PASS |
| `BGN1` | `JP` | PASS | PASS | **FAIL** — `EOID LEBGR1e003Tp4N3J` duplicates `BG01` (`status = -1`) |

`BG01` / `BG02` do exercise the `Country starts_with EU-list` branch (their country
codes are in the list), but the `THEN` check passes too, so the result is still PASS.

### Seeing the join change a result

With this ruleset the join is wired but **invisible** — no active rule depends on a
location-only field. To watch it flip PASS/FAIL, add a rule that checks one, e.g. a
Data-Type `Exist` check on `City`: it fails when locations aren't loaded and passes
once they are.

### Other notes

- `passed = 1/true` also means "rule not applicable" (a `Trigger` / `IF` was false).
- `status` / `description` are `NaN` on every passing row; only populated on failure.
- Every POST inserts **and validates** all entities — there is no skip/duplicate check
  (`entity_hash` is just `sha256(systemCode, businessEntityCode)`). Re-POSTing the same
  payload adds another full set of rows to `entities` and `validation_results`; run
  `reset_db.ipynb` between runs to keep the tables tidy.
- Locations must be POSTed **before** the entities (the join runs during insert).